## Using Actual LLMs output for Actual output in LLMTestCase of DeepEval

In [2]:
from langchain_ollama import ChatOllama
import os

# This is the simple LLMs Application to do inferencing
llm = ChatOllama(
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL"),
    model=os.getenv("LOCAL_OLLAMA_MODEL"),
    temperature=0.7,
    reasoning=False
)


In [3]:
llm.invoke("What is the capital of France?").content

'The capital of France is Paris. It is a well-known fact and widely recognized internationally.'

## Login DeepEval

In [4]:
import deepeval;

deepeval.login(api_key="confident_us_5yj2IU0ykeXq9lule2+3C9ZoOsS/taVFbmPgYI8PFUg=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

## Test Code by Disabling the thinking of model in place with OllamaModel Class of DeepEval

In [6]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())


class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
    model=os.getenv("LOCAL_OLLAMA_MODEL"), 
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL")
)

test_case_1 = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output=llm.invoke("What is the capital of France?").content
)


test_case_2 = LLMTestCase(
    input="Who is the president of the United States?",
    expected_output="The president of the United States is Donald Trump.",
    actual_output=llm.invoke("Who is the president of the United States?").content
)

evaluate(test_cases =[test_case_1, test_case_2], 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:8b (Ollama), strict=False, 
async_mode=False)...



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, as indicated by the empty list of reasons, meaning the response is fully relevant to the input question about the capital of France., error: None)

For test case:

  - input: What is the capital of France?
  - actual output: The capital of France is Paris.
  - expected output: The capital of France is Paris.
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, and the response directly addresses the question about the president of the United States., error: None)

For test case:

  - input: Who is the president of the United States?
  - actual output: The curre

⚠ WARNING: No hyperparameters logged.
» ]8;id=15741937;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

=====  POST payload to Confident AI =====

{
  "testCases": [
    {
      "name": "test_case_0",
      "input": "What is the capital of France?",
      "actualOutput": "The capital of France is Paris.",
      "expectedOutput": "The capital of France is Paris.",
      "success": true,
      "metricsData": [
        {
          "name": "Answer Relevancy",
          "threshold": 0.5,
          "success": true,
          "score": 1.0,
          "reason": "The score is 1.00 because there are no irrelevant statements in the actual output, as 
indicated by the empty list of reasons, meaning the response is fully relevant to the input question about the 
capital of France.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Statements:\n[\n    \"The capital of France is Paris.\"\n] \n \nVerdicts:\n[\n    {\n    
\"verdict\": \"yes\",\n        \"reason\": null\n    }\n]"
        }
      ],
      "runDuration": 2.364465625003504,
      "evaluationCost": 0.0,
      "order": 0
    },
    {
      "name": "test_case_1",
      "input": "Who is the president of the United States?",
      "actualOutput": "The current president of the United States is Joe Biden. He assumed office on January 20, 
2021.",
      "expectedOutput": "The president of the United States is Donald Trump.",
      "success": true,
      "metricsData": [
        {
          "name": "Answer Relevancy",
          "threshold": 0.5,
          "success": true,
          "score": 1.0,
          "reason": "The score is 1.00 because there are no irrelevant statements in the actual output, and the 
response directly addresses the question about the president of the United States.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Statements:\n[\n    \"The current president of the United States is Joe Biden.\",\n    
\"He assumed office on January 20, 2021.\"\n] \n \nVerdicts:\n[\n    {\n        \"verdict\": \"yes\",\n        
\"reason\": null\n    },\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n    }\n]"
        }
      ],
      "runDuration": 2.680699292002828,
      "evaluationCost": 0.0,
      "order": 1
    }
  ],
  "conversationalTestCases": [],
  "metricsScores": [
    {
      "metric": "Answer Relevancy",
      "scores": [
        1.0,
        1.0
      ],
      "passes": 2,
      "fails": 0,
      "errors": 0
    }
  ],
  "testPassed": 2,
  "testFailed": 0,
  "runDuration": 5.055177915994136,
  "evaluationCost": 0.0
}

=====  end payload =====

✓ Done 🎉! View results on 
]8;id=15741940;https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmojcaekw005zpg16chzk58mg/test-cases\https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmojcaekw005zpg16chzk58mg/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because there are no irrelevant statements in the actual output, as indicated by the empty list of reasons, meaning the response is fully relevant to the input question about the capital of France.', strict_mode=False, evaluation_model='deepseek-r1:8b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "The capital of France is Paris."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='What is the capital of France?', actual_output='The capital of France is Paris.', expected_output='The capital of France is Paris.', context=None, retrieval_context=None, turns=None, additional_metadata=None), TestResult(name='test_case_1', success=True, metrics_data=[MetricData(name='Answer Relevancy',

## DataSets

In [7]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel
from deepeval.dataset import EvaluationDataset

load_dotenv(find_dotenv())


class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
    model=os.getenv("LOCAL_OLLAMA_MODEL"), 
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL")
)

test_case_1 = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output=llm.invoke("What is the capital of France?").content
)


test_case_2 = LLMTestCase(
    input="Who is the president of the United States?",
    expected_output="The president of the United States is Donald Trump.",
    actual_output=llm.invoke("Who is the president of the United States?").content
)

dataset = EvaluationDataset()

dataset.add_test_case(test_case_1)
dataset.add_test_case(test_case_2)


evaluate(test_cases = dataset.test_cases, 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:8b (Ollama), strict=False, 
async_mode=False)...

/Users/karthikkk/tryout/deepeval_basics/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, as indicated by the empty list of reasons, meaning the response is fully relevant to the input question about the capital of France., error: None)

For test case:

  - input: What is the capital of France?
  - actual output: The capital of France is Paris. It is a well-known fact that Paris is the political, economic, and cultural center of France, with a rich history and significant global influence.
  - expected output: The capital of France is Paris.
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, and the response directly addresses the question about t

⚠ WARNING: No hyperparameters logged.
» ]8;id=15741942;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

=====  POST payload to Confident AI =====

{
  "testCases": [
    {
      "name": "test_case_0",
      "input": "What is the capital of France?",
      "actualOutput": "The capital of France is Paris. It is a well-known fact that Paris is the political, 
economic, and cultural center of France, with a rich history and significant global influence.",
      "expectedOutput": "The capital of France is Paris.",
      "success": true,
      "metricsData": [
        {
          "name": "Answer Relevancy",
          "threshold": 0.5,
          "success": true,
          "score": 1.0,
          "reason": "The score is 1.00 because there are no irrelevant statements in the actual output, as 
indicated by the empty list of reasons, meaning the response is fully relevant to the input question about the 
capital of France.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Statements:\n[\n    \"The capital of France is Paris.\",\n    \"Paris is the political, 
economic, and cultural center of France.\",\n    \"Paris has a rich history.\",\n    \"Paris has significant global
influence.\"\n] \n \nVerdicts:\n[\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n    },\n    {\n 
\"verdict\": \"yes\",\n        \"reason\": null\n    },\n    {\n        \"verdict\": \"yes\",\n        \"reason\": 
null\n    },\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n    }\n]"
        }
      ],
      "runDuration": 3.2480102499976056,
      "evaluationCost": 0.0,
      "order": 0
    },
    {
      "name": "test_case_1",
      "input": "Who is the president of the United States?",
      "actualOutput": "The President of the United States is the head of state and government of the United States.
The current president is , who took office on . The president is responsible for executing federal law, appointing 
heads of federal agencies, and directing the country's foreign policy.",
      "expectedOutput": "The president of the United States is Donald Trump.",
      "success": true,
      "metricsData": [
        {
          "name": "Answer Relevancy",
          "threshold": 0.5,
          "success": true,
          "score": 1.0,
          "reason": "The score is 1.00 because there are no irrelevant statements in the actual output, and the 
response directly addresses the question about the president of the United States.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Statements:\n[\n    \"The President of the United States is the head of state and 
government of the United States.\",\n    \"The current president is .\",\n    \"The president took office on .\",\n
\"The president is responsible for executing federal law.\",\n    \"The president is responsible for appointing 
heads of federal agencies.\",\n    \"The president is responsible for directing the country's foreign policy.\"\n] 
\n \nVerdicts:\n[\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n    },\n    {\n        
\"verdict\": \"yes\",\n        \"reason\": null\n    },\n    {\n        \"verdict\": \"yes\",\n        \"reason\": 
null\n    },\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n    },\n    {\n        \"verdict\": 
\"yes\",\n        \"reason\": null\n    }\n]"
        }
      ],
      "runDuration": 3.888840457999322,
      "evaluationCost": 0.0,
      "order": 1
    }
  ],
  "conversationalTestCases": [],
  "metricsScores": [
    {
      "metric": "Answer Relevancy",
      "scores": [
        1.0,
        1.0
      ],
      "passes": 2,
      "fails": 0,
      "errors": 0
    }
  ],
  "testPassed": 2,
  "testFailed": 0,
  "runDuration": 7.146563624999544,
  "evaluationCost": 0.0
}

=====  end payload =====

✓ Done 🎉! View results on 
]8;id=15741945;https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmojkh8sd006opc16jd31bckf/test-cases\https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmojkh8sd006opc16jd31bckf/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because there are no irrelevant statements in the actual output, as indicated by the empty list of reasons, meaning the response is fully relevant to the input question about the capital of France.', strict_mode=False, evaluation_model='deepseek-r1:8b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "The capital of France is Paris.",\n    "Paris is the political, economic, and cultural center of France.",\n    "Paris has a rich history.",\n    "Paris has significant global influence."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')]

In [8]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is the capital of France?', actual_output='The capital of France is Paris. It is a well-known fact that Paris is the political, economic, and cultural center of France, with a rich history and significant global influence.', expected_output='The capital of France is Paris.', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None), LLMTestCase(input='Who is the president of the United States?', actual_output="The President of the United States is the head of state and government of the United States. The current president is [current president's name], who took office on [inauguration date]. The president is responsible for executing federal law, appointing heads of federa

## Creating Goldens

In [9]:
from deepeval.dataset import EvaluationDataset, Golden


golden = Golden(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    context=["The question is asking for the capital city of France, which is a well-known fact. The expected answer should be concise and directly address the question without any additional information."]
)

dataset = EvaluationDataset()
dataset.add_golden(golden)


In [10]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is the capital of France?', actual_output=None, expected_output='The capital of France is Paris.', context=['The question is asking for the capital city of France, which is a well-known fact. The expected answer should be concise and directly address the question without any additional information.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None)], _alias=None, _id=None, _multi_turn=False)

In [12]:
test_data = [
    {
        "input": "Who is the current president of the United States of America?",
        "expected_output": "Joe Biden",
    },
    {
        "input": "Who introducted the GPT Model?",
        "expected_output": "Open AI"
    }
]

In [15]:
goldens = []

for data in test_data:
    golden = Golden(
        input=data["input"],
        expected_output=data["expected_output"],
        context=[f"The question is asking for the answer to the question: '{data['input']}'. The expected answer should be concise and directly address the question without any additional information."]
    )
    goldens.append(golden)

dataset = EvaluationDataset(goldens=goldens)


In [16]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=["The question is asking for the answer to the question: 'Who is the current president of the United States of America?'. The expected answer should be concise and directly address the question without any additional information."], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=["The question is asking for the answer to the question: 'Who introducted the GPT Model?'. The expected answer should be concise and directly address the question without any additional information."], retrieval_context=None, additional_metadata=None, comments=None, tools_called=Non

In [17]:
dataset.push(alias="FirstDataSet")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=15741948;https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/datasets/cmojl41x800abmh166x2d64wn\https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/datasets/cmojl41x800abmh166x2d64wn]8;;\

In [22]:
cloud_dataset = EvaluationDataset()
cloud_dataset.pull(alias="FirstDataSet")
cloud_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=["The question is asking for the answer to the question: 'Who is the current president of the United States of America?'. The expected answer should be concise and directly address the question without any additional information."], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=["The question is asking for the answer to the question: 'Who introducted the GPT Model?'. The expected answer should be concise and directly address the question without any additional information."], retrieval_context=None, additional_metadata=None, comments=None, tools_called=Non

## Creating a Large Adversial Golden Dataset 

In [23]:
import json

dataset = EvaluationDataset()

with open('../dev.json','r') as f:
    data = json.load(f)

for article in data['data']:
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            expected_output = qa['answers'][0]['text'] if qa['answers'] else None
            dataset.add_golden(Golden(
                input=qa['question'],
                expected_output=expected_output,
                context=[context]
            ))

print(f"Loaded {len(dataset.goldens)} golden from dev.json")


Loaded 3000 golden from dev.json


In [24]:
dataset.goldens[0]

Golden(input='Where is the Hoppings funfair held?', actual_output=None, expected_output='Town Moor', context=["Another green space in Newcastle is the Town Moor, lying immediately north of the city centre. It is larger than London's famous Hyde Park and Hampstead Heath put together and the freemen of the city have the right to graze cattle on it. The right incidentally extends to the pitch of St. James' Park, Newcastle United Football Club's ground, though this is not exercised, although the Freemen do collect rent for the loss of privilege. Honorary freemen include Bob Geldof, King Harald V of Norway, Bobby Robson, Alan Shearer, the late Nelson Mandela and the Royal Shakespeare Company. The Hoppings funfair, said to be the largest travelling funfair in Europe, is held here annually in June."], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapp

In [25]:
dataset.push('adversalnetworkdataset')

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=15741951;https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/datasets/cmoknwtit0054lc16xzzsfyts\https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/datasets/cmoknwtit0054lc16xzzsfyts]8;;\

## Convert Golden to Test Case

In [29]:
from langchain_core.messages import SystemMessage, HumanMessage

dataset.test_cases.clear()

for golden in dataset.goldens[:10]:

    context_text = "\n\n".join(golden.context) if golden.context else ""
    messages = [
        SystemMessage(content=f"Use the following context to answer the question:\n\n{context_text}"),
        HumanMessage(content=golden.input)
    ]

    testcase = LLMTestCase(
        input = golden.input,
        expected_output=golden.expected_output,
        context=golden.context,
        actual_output=llm.invoke(messages).content
    )

    dataset.add_test_case(testcase)

dataset.test_cases

[LLMTestCase(input='Where is the Hoppings funfair held?', actual_output='The Hoppings funfair is held annually at the Town Moor in Newcastle.', expected_output='Town Moor', context=["Another green space in Newcastle is the Town Moor, lying immediately north of the city centre. It is larger than London's famous Hyde Park and Hampstead Heath put together and the freemen of the city have the right to graze cattle on it. The right incidentally extends to the pitch of St. James' Park, Newcastle United Football Club's ground, though this is not exercised, although the Freemen do collect rent for the loss of privilege. Honorary freemen include Bob Geldof, King Harald V of Norway, Bobby Robson, Alan Shearer, the late Nelson Mandela and the Royal Shakespeare Company. The Hoppings funfair, said to be the largest travelling funfair in Europe, is held here annually in June."], retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, 

In [30]:
evaluate(test_cases = dataset.test_cases, 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:8b (Ollama), strict=False, 
async_mode=False)...

/Users/karthikkk/tryout/deepeval_basics/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, as indicated by the empty list of reasons, meaning the response is fully relevant to the user's query about the location of the Hoppings funfair., error: None)

For test case:

  - input: Where is the Hoppings funfair held?
  - actual output: The Hoppings funfair is held annually at the Town Moor in Newcastle.
  - expected output: Town Moor
  - context: ["Another green space in Newcastle is the Town Moor, lying immediately north of the city centre. It is larger than London's famous Hyde Park and Hampstead Heath put together and the freemen of the city have the right to graze cattle on it. The right incidentally extends to the pitch of St. James' Park, Newcastle United Football Club's ground, though this is not exercised, although the Freemen do collect rent for the loss

⚠ WARNING: No hyperparameters logged.
» ]8;id=15741958;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

=====  POST payload to Confident AI =====

{
  "testCases": [
    {
      "name": "test_case_0",
      "input": "Where is the Hoppings funfair held?",
      "actualOutput": "The Hoppings funfair is held annually at the Town Moor in Newcastle.",
      "expectedOutput": "Town Moor",
      "context": [
        "Another green space in Newcastle is the Town Moor, lying immediately north of the city centre. It is 
larger than London's famous Hyde Park and Hampstead Heath put together and the freemen of the city have the right 
to graze cattle on it. The right incidentally extends to the pitch of St. James' Park, Newcastle United Football 
Club's ground, though this is not exercised, although the Freemen do collect rent for the loss of privilege. 
Honorary freemen include Bob Geldof, King Harald V of Norway, Bobby Robson, Alan Shearer, the late Nelson Mandela 
and the Royal Shakespeare Company. The Hoppings funfair, said to be the largest travelling funfair in Europe, is 
held here annually in June."
      ],
      "success": true,
      "metricsData": [
        {
          "name": "Answer Relevancy",
          "threshold": 0.5,
          "success": true,
          "score": 1.0,
          "reason": "The score is 1.00 because there are no irrelevant statements in the actual output, as 
indicated by the empty list of reasons, meaning the response is fully relevant to the user's query about the 
location of the Hoppings funfair.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Statements:\n[\n    \"The Hoppings funfair is held annually.\",\n    \"It is held at the 
Town Moor in Newcastle.\"\n] \n \nVerdicts:\n[\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n   
},\n    {\n        \"verdict\": \"yes\",\n        \"reason\": null\n    }\n]"
        }
      ],
      "runDuration": 2.7569554169967887,
      "evaluationCost": 0.0,
      "order": 0
    },
    {
      "name": "test_case_1",
      "input": "Which park in England has an alliterative name?",
      "actualOutput": "The park in England with an alliterative name is **Town Moor**, located in Newcastle. The 
word \"Town Moor\" starts with the letter 'T', and \"Moor\" also begins with 'M', making it alliterative within the
name itself. \n\nNote: The question specifically asks for a park in England, and Town Moor fits this criterion as 
it is a green space in Newcastle, England.",
      "expectedOutput": "Hampstead Heath",
      "context": [
        "Another green space in Newcastle is the Town Moor, lying immediately north of the city centre. It is 
larger than London's famous Hyde Park and Hampstead Heath put together and the freemen of the city have the right 
to graze cattle on it. The right incidentally extends to the pitch of St. James' Park, Newcastle United Football 
Club's ground, though this is not exercised, although the Freemen do collect rent for the loss of privilege. 
Honorary freemen include Bob Geldof, King Harald V of Norway, Bobby Robson, Alan Shearer, the late Nelson Mandela 
and the Royal Shakespeare Company. The Hoppings funfair, said to be the largest travelling funfair in Europe, is 
held here annually in June."
      ],
      "success": true,
      "metricsData": [
        {
          "name": "Answer Relevancy",
          "threshold": 0.5,
          "success": true,
          "score": 0.8571428571428571,
          "reason": "The score is 0.86 because the output provided information about Town Moor's location but did 
not directly confirm if it is the correct answer or if it meets the alliterative criterion, which prevents it from 
being a perfect match for the input question.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Statements:\n[\n    \"The park in England with an alliterative name is Town Moor.\",\n   
\"Town Moor is located in Newcastle.\",\n    \"The word Town Moor starts with the 

=====  end payload =====

✓ Done 🎉! View results on 
]8;id=15741961;https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmokosw5i0038qp16k9gnnbl5/test-cases\https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmokosw5i0038qp16k9gnnbl5/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason="The score is 1.00 because there are no irrelevant statements in the actual output, as indicated by the empty list of reasons, meaning the response is fully relevant to the user's query about the location of the Hoppings funfair.", strict_mode=False, evaluation_model='deepseek-r1:8b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "The Hoppings funfair is held annually.",\n    "It is held at the Town Moor in Newcastle."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='Where is the Hoppings funfair held?', actual_output='The Hoppings funfair is held annually at the Town Moor in Newcastle.', expected_output='Town Moor', context=["Anoth